# Generation of commication themes per segment

In [2]:
import os
import json
import pandas as pd
from typing import List
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_ollama import ChatOllama
from dotenv import load_dotenv

load_dotenv()

# --- 1. CONFIGURATION ---
INPUT_THEMES_JSON = "./output/segment_themes.json" 
INPUT_MATRIX_JSON = "./output/allowed_tone_hook_matrix.json"
OUTPUT_CSV = "./output/communication_themes.csv"
ollama_model = "llama3.2:latest"

# --- Ollama connection (switch OLLAMA_MODE in .env: "web" or "local") ---
_mode = os.getenv("OLLAMA_MODE", "local")
OLLAMA_BASE_URL = (
    os.getenv("OLLAMA_WEB_URL", "https://unsympathizing-censorian-buena.ngrok-free.dev")
    if _mode == "web"
    else os.getenv("OLLAMA_LOCAL_URL", "http://127.0.0.1:11434")
)
print(f"Ollama mode: {_mode.upper()} → {OLLAMA_BASE_URL}")

# --- 2. DEFINE STRICT PYDANTIC SCHEMA ---
class SegmentCommunication(BaseModel):
    tone_preferences: List[str] = Field(description="2-3 specific tones chosen ONLY from the allowed_tones list.")
    hooks_per_segment: List[str] = Field(description="1-2 specific hooks chosen from the hook_taxonomy.")

# --- 3. LLM SETUP ---
llm = ChatOllama(model=ollama_model, temperature=0, format="json", base_url=OLLAMA_BASE_URL)
parser = JsonOutputParser(pydantic_object=SegmentCommunication)

def get_segment_communication(segment_name, primary_theme, secondary_theme, matrix_data):
    raw_hooks = matrix_data.get('hook_taxonomy', [])
    available_hooks = [h.get("hook_name", h.get("name", str(h))) if isinstance(h, dict) else str(h) for h in raw_hooks]

    system_prompt = (
        "You are an expert behavioral product marketer. Assign tones and hooks to a user segment.\n\n"
        f"Allowed Tones: {matrix_data.get('allowed_tones', [])}\n"
        f"Disallowed Tones: {matrix_data.get('disallowed_tones', [])}\n"
        f"Available Hooks: {available_hooks}\n\n"
        f"{parser.get_format_instructions()}"
    )
    
    user_prompt = (
        f"Segment Name: {segment_name}\n"
        f"Primary Theme: {primary_theme}\n"
        f"Secondary Theme: {secondary_theme}\n\n"
        "Based on these Octalysis themes, select the most appropriate tones and hooks from the available lists."
    )
    
    res = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)])
    try: 
        return parser.parse(res.content)
    except Exception as e: 
        print(f"Error parsing {segment_name}: {e}")
        return {"tone_preferences": ["Error"], "hooks_per_segment": ["Error"]}

def generate_communication_themes():
    print("Loading data directly from segment_themes.json...")
    
    with open(INPUT_THEMES_JSON, "r", encoding="utf-8") as f:
        themes_data = json.load(f)
        
    with open(INPUT_MATRIX_JSON, "r", encoding="utf-8") as f:
        matrix_data = json.load(f)

    results = []
    print(f"Processing {len(themes_data.get('segments', []))} segments...")
    
    for segment in themes_data.get('segments', []):
        seg_name = segment.get('segment_name', 'Unknown')
        sorted_drives = sorted(
            segment.get('octolysis_themes', []), 
            key=lambda x: x.get('relevance_score', 0), 
            reverse=True
        )
        primary_theme = sorted_drives[0]['core_drive'] if len(sorted_drives) > 0 else "None"
        secondary_theme = sorted_drives[1]['core_drive'] if len(sorted_drives) > 1 else "None"
        
        print(f"Mapping {seg_name} (Primary: {primary_theme}, Secondary: {secondary_theme})...")
        llm_result = get_segment_communication(seg_name, primary_theme, secondary_theme, matrix_data)
        
        raw_tones = llm_result.get("tone_preferences", [])
        raw_hooks = llm_result.get("hooks_per_segment", [])
        safe_tones = [t.get("tone_name", str(t)) if isinstance(t, dict) else str(t) for t in raw_tones]
        safe_hooks = [h.get("hook_name", str(h)) if isinstance(h, dict) else str(h) for h in raw_hooks]

        results.append({
            "Segment ID": segment.get('segment_id', ''),
            "Segment Name": seg_name,
            "Primary Theme": primary_theme,
            "Secondary Theme": secondary_theme,
            "Tone Preferences": ", ".join(safe_tones),
            "Hooks per Segment": ", ".join(safe_hooks)
        })

    df = pd.DataFrame(results)
    os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
    df.to_csv(OUTPUT_CSV, index=False)
    df.to_csv("../iteration_0_before_learning/communication_themes.csv", index=False)
    
    print(f"\n✓ Success! Saved enriched mapping to {OUTPUT_CSV}")
    print("\nPreview:")
    print(df[['Segment Name', 'Primary Theme', 'Hooks per Segment']].head())

if __name__ == "__main__":
    generate_communication_themes()


Ollama mode: WEB → https://unsympathizing-censorian-buena.ngrok-free.dev/
Loading data directly from segment_themes.json...
Processing 9 segments...
Mapping Streak Champions (Primary: Accomplishment, Secondary: Loss Avoidance)...
Mapping Power Learners (Primary: Accomplishment, Secondary: Loss Avoidance)...
Mapping Top Competitors (Primary: Accomplishment, Secondary: Social Influence)...
Mapping Coin Collectors (Primary: Accomplishment, Secondary: Ownership)...
Mapping Steady Students (Primary: Accomplishment, Secondary: Social Influence)...
Mapping Social Observers (Primary: Social Influence, Secondary: Accomplishment)...
Mapping Fading Players (Primary: Loss Avoidance, Secondary: Social Influence)...
Mapping Stuck Users (Primary: Loss Avoidance, Secondary: Accomplishment)...
Mapping Disconnected (Primary: Loss Avoidance, Secondary: Social Influence)...

✓ Success! Saved enriched mapping to ./output/communication_themes.csv

Preview:
       Segment Name   Primary Theme               H

# Generating Templates

In [3]:
import uuid
import pandas as pd
import json
import os
from typing import List
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_ollama import ChatOllama
from dotenv import load_dotenv

load_dotenv()

# ==========================================
# Step 1: Define the Strict Output Schema
# ==========================================
class MessageTemplate(BaseModel):
    hook_used: str = Field(description="The psychological hook or angle used in the message.")
    feature_reference: str = Field(description="The specific app feature referenced (e.g., Daily Streak, Leaderboard).")
    content_english: str = Field(description="The English push notification content.")
    content_hindi: str = Field(description="The precise Hindi translation of the push notification.")

class TemplateResponse(BaseModel):
    templates: List[MessageTemplate] = Field(
        description="A list containing EXACTLY 5 distinct message templates. No more, no less.",
        min_length=5,
        max_length=5
    )

# ==========================================
# Step 2: Configuration & Model Init
# ==========================================
PATH_SEGMENT_GOALS = './output/segment_lifecycle_goals.csv'
PATH_COMM_THEMES = './output/communication_themes.csv' 
PATH_OUTPUT_CSV = './output/message_templates.csv'
PATH_NORTH_STAR = './output/company_north_star.json'

# --- Ollama connection (switch OLLAMA_MODE in .env: "web" or "local") ---
_mode = os.getenv("OLLAMA_MODE", "local")
OLLAMA_BASE_URL = (
    os.getenv("OLLAMA_WEB_URL", "https://unsympathizing-censorian-buena.ngrok-free.dev")
    if _mode == "web"
    else os.getenv("OLLAMA_LOCAL_URL", "http://127.0.0.1:11434")
)
print(f"Ollama mode: {_mode.upper()} → {OLLAMA_BASE_URL}")

base_llm = ChatOllama(model="llama3.2:latest", temperature=0.3, base_url=OLLAMA_BASE_URL)
structured_llm = base_llm.with_structured_output(TemplateResponse)

# Exact JSON schema shown to the LLM to eliminate ambiguity
RESPONSE_FORMAT_EXAMPLE = """{
  "templates": [
    {
      "hook_used": "<hook name>",
      "feature_reference": "<app feature>",
      "content_english": "<push notification text in English>",
      "content_hindi": "<push notification text in Hindi>"
    },
    { ... },
    { ... },
    { ... },
    { ... }
  ]
}"""

# ==========================================
# Step 3: Data Preparation (The Cross-Join)
# ==========================================
def prepare_tasks():
    df_goals = pd.read_csv(PATH_SEGMENT_GOALS)[['segment_id', 'lifecycle_stage', 'primary_goal']]
    df_themes = pd.read_csv(PATH_COMM_THEMES)
    df_themes.rename(columns={'Segment ID': 'segment_id'}, inplace=True)
    df_merged = pd.merge(df_goals, df_themes, on='segment_id')
    df_primary = df_merged.copy()
    df_primary['theme'] = df_primary['Primary Theme']
    df_secondary = df_merged.copy()
    df_secondary['theme'] = df_secondary['Secondary Theme']
    df_tasks = pd.concat([df_primary, df_secondary], ignore_index=True)
    df_tasks.drop(columns=['Primary Theme', 'Secondary Theme', 'Segment Name'], inplace=True, errors='ignore')
    return df_tasks.to_dict(orient='records')

# ==========================================
# Step 4: LLM Generator & Validation
# ==========================================
try:
    with open(PATH_NORTH_STAR, 'r', encoding='utf-8') as f:
        ns_data = json.load(f)
        NORTH_STAR_METRIC = ns_data.get('inferred_north_star', 'Daily Active Usage')
except FileNotFoundError:
    print(f"Warning: {PATH_NORTH_STAR} not found. Using fallback metric.")
    NORTH_STAR_METRIC = "Daily Active Usage"

def generate_templates_for_row(llm_instance, row):
    system_prompt = (
        f"You are an expert copywriter for a digital product. Our North Star metric is {NORTH_STAR_METRIC}.\n\n"
        "CRITICAL OUTPUT RULE: You MUST return EXACTLY 5 templates in the 'templates' array — "
        "never 4, never 6, always exactly 5. Each template must be unique.\n\n"
        f"Your response must match this exact JSON structure:\n{RESPONSE_FORMAT_EXAMPLE}"
    )
    user_prompt = (
        f"Write EXACTLY 5 (five) push notification templates for a user in the '{row['lifecycle_stage']}' stage.\n"
        f"Their current goal is: {row['primary_goal']}.\n\n"
        "Behavioral design constraints — you MUST respect all of these:\n"
        f"- Octalysis Core Drive: {row['theme']}\n"
        f"- Required Tones: {row['Tone Preferences']}\n"
        f"- Suggested Hooks to leverage: {row['Hooks per Segment']}\n\n"
        "Remember: the 'templates' array must contain EXACTLY 5 objects. Count them before returning."
    )
    messages = [SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)]
    try:
        result = llm_instance.invoke(messages)
        if not result or len(result.templates) != 5:
            print(f"⚠ Validation failed for {row['segment_id']} - {row['theme']}: Returned {len(result.templates) if result else 0} items.")
            return []
        enriched_templates = []
        for tpl in result.templates:
            enriched_templates.append({
                "template_id": f"TPL-{uuid.uuid4().hex[:8].upper()}",
                "segment_id": row['segment_id'],
                "lifecycle_stage": row['lifecycle_stage'],
                "goal": row['primary_goal'],
                "theme": row['theme'],
                "tone_preferences": row['Tone Preferences'],
                "hook_used": tpl.hook_used,
                "feature_reference": tpl.feature_reference,
                "content_english": tpl.content_english,
                "content_hindi": tpl.content_hindi
            })
        return enriched_templates
    except Exception as e:
        print(f"✗ Generation Error for {row['segment_id']} - {row['theme']}: {str(e)}")
        return []

# ==========================================
# Step 5: Execution & Export
# ==========================================
def main():
    tasks_data = prepare_tasks()
    print(f"Prepared {len(tasks_data)} distinct prompt tasks.")
    all_results = []
    for index, row in enumerate(tasks_data):
        print(f"Processing task {index + 1}/{len(tasks_data)}: {row['segment_id']} | {row['lifecycle_stage']} | {row['theme']}")
        result = generate_templates_for_row(structured_llm, row)
        all_results.extend(result)
    print(f"\nSuccessfully generated {len(all_results)} valid templates.")
    if all_results:
        df_final = pd.DataFrame(all_results)
        cols = ['template_id', 'segment_id', 'lifecycle_stage', 'goal', 'theme', 'tone_preferences',
                'hook_used', 'feature_reference', 'content_english', 'content_hindi']
        valid_cols = [c for c in cols if c in df_final.columns]
        df_final = df_final[valid_cols]
        os.makedirs(os.path.dirname(PATH_OUTPUT_CSV), exist_ok=True)
        df_final.to_csv(PATH_OUTPUT_CSV, index=False)
        df_final.to_csv("../iteration_0_before_learning/message_templates.csv", index=False)
        print(f"Exported to {PATH_OUTPUT_CSV} successfully.")
    else:
        print("No templates generated.")

if __name__ == "__main__":
    main()


Ollama mode: WEB → https://unsympathizing-censorian-buena.ngrok-free.dev/
Prepared 72 distinct prompt tasks.
Processing task 1/72: S4 | paid | Accomplishment
Processing task 2/72: S4 | churned | Accomplishment
Processing task 3/72: S4 | trial | Accomplishment
Processing task 4/72: S4 | inactive | Accomplishment
Processing task 5/72: S7 | paid | Loss Avoidance
Processing task 6/72: S7 | churned | Loss Avoidance
Processing task 7/72: S7 | trial | Loss Avoidance
Processing task 8/72: S7 | inactive | Loss Avoidance
Processing task 9/72: S6 | paid | Social Influence
Processing task 10/72: S6 | churned | Social Influence
Processing task 11/72: S6 | trial | Social Influence
Processing task 12/72: S6 | inactive | Social Influence
Processing task 13/72: S5 | paid | Accomplishment
Processing task 14/72: S5 | churned | Accomplishment
Processing task 15/72: S5 | trial | Accomplishment
Processing task 16/72: S5 | inactive | Accomplishment
Processing task 17/72: S9 | paid | Loss Avoidance
Processing

# Timing window finder using GMM

In [4]:
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
import warnings
import os

# Suppress warnings from GMM when clusters are tiny
warnings.filterwarnings('ignore')

# ==========================================
# Configuration & File Paths
# ==========================================
PATH_USER_SEGMENTS_CSV = './output/users_segmented.csv'
PATH_OUTPUT_RECOMMENDATIONS = './output/timing_recommendations.csv'

# ==========================================
# 1. Standard Window Mapping
# ==========================================
def map_hour_to_window(hour: float) -> str:
    """Maps a 0-23 hour float/int to the 6 predefined time windows."""
    if pd.isna(hour):
        return 'afternoon' # Safe default
        
    hour = int(round(hour)) % 24
    if 6 <= hour < 9: return 'early_morning'
    elif 9 <= hour < 12: return 'mid_morning'
    elif 12 <= hour < 15: return 'afternoon'
    elif 15 <= hour < 18: return 'late_afternoon'
    elif 18 <= hour < 21: return 'evening'
    else: return 'night'

# ==========================================
# 2. ML Optimizer Class
# ==========================================
class DynamicWindowOptimizer:
    def __init__(self, max_windows: int = 3):
        self.max_windows = max_windows

    def fit(self, csv_path: str = PATH_USER_SEGMENTS_CSV):
        """Runs the ML algorithm to determine optimal windows and expected CTR."""
        df = pd.read_csv(csv_path)
        hour_col = 'preferred_hour' if 'preferred_hour' in df.columns else 'hour'
        df_clean = df.dropna(subset=[hour_col])
        
        # Pre-calculate the mapped window for every user to easily look up historical CTR
        df_clean['assigned_window'] = df_clean[hour_col].apply(map_hour_to_window)
        
        recommendations = []
        
        for segment_id, group in df_clean.groupby('segment_id'):
            segment_name = group['segment_name'].iloc[0] if 'segment_name' in group.columns else 'Unknown'
            X = group[hour_col].values.reshape(-1, 1)
            
            # --- Edge Case: Very few users ---
            if len(X) < 5:
                mean_hour = X.mean()
                window = map_hour_to_window(mean_hour)
                expected_ctr = group['notif_open_rate_30d'].mean() if 'notif_open_rate_30d' in group.columns else 0.0
                expected_activation = group['activation_score'].mean() if 'activation_score' in group.columns else 0.0
                
                recommendations.append({
                    'segment_id': segment_id,
                    'segment_name': segment_name,
                    'optimal_window': window,
                    'peak_hour': round(mean_hour, 1),
                    'user_fraction': 1.0, # 100% of this tiny segment
                    'expected_ctr': round(expected_ctr, 3),
                    'expected_engagement_score': round(expected_activation, 3)
                })
                continue

            # --- GMM Optimization ---
            best_gmm = None
            lowest_bic = np.inf
            max_k_to_try = min(self.max_windows, len(np.unique(X))) 
            
            for k in range(1, max_k_to_try + 1):
                gmm = GaussianMixture(n_components=k, random_state=42)
                gmm.fit(X)
                bic_score = gmm.bic(X)
                if bic_score < lowest_bic:
                    lowest_bic = bic_score
                    best_gmm = gmm
            
            # --- Extract Recommendations from Best Model ---
            for i in range(best_gmm.n_components):
                cluster_hour = best_gmm.means_[i][0]
                cluster_weight = best_gmm.weights_[i]
                window_name = map_hour_to_window(cluster_hour)
                
                # Look up historical engagement for users in this segment who actually prefer this window
                users_in_window = group[group['assigned_window'] == window_name]
                
                # Calculate expected CTR/Engagement (Fallback to segment mean if window is currently empty)
                if len(users_in_window) > 0:
                    expected_ctr = users_in_window['notif_open_rate_30d'].mean()
                    expected_engagement = users_in_window['activation_score'].mean()
                else:
                    expected_ctr = group['notif_open_rate_30d'].mean()
                    expected_engagement = group['activation_score'].mean()
                
                recommendations.append({
                    'segment_id': segment_id,
                    'segment_name': segment_name,
                    'optimal_window': window_name,
                    'peak_hour': round(cluster_hour, 1),
                    'user_fraction': round(cluster_weight, 3), # E.g., 0.65 means 65% of the segment prefers this window
                    'expected_ctr': round(expected_ctr, 3),
                    'expected_engagement_score': round(expected_engagement, 3)
                })

        # Convert to DataFrame and sort by Segment ID and User Fraction (descending)
        df_recommendations = pd.DataFrame(recommendations)
        df_recommendations = df_recommendations.sort_values(by=['segment_id', 'user_fraction'], ascending=[True, False])
        
        # Deduplicate identical windows within the same segment (GMM sometimes splits close hours that map to the same text window)
        df_recommendations = df_recommendations.drop_duplicates(subset=['segment_id', 'optimal_window'], keep='first')
        
        return df_recommendations

# ==========================================
# 3. Execution & Export
# ==========================================
if __name__ == "__main__":
    ml_optimizer = DynamicWindowOptimizer(max_windows=3)
    
    try:
        print(f"Loading data from {PATH_USER_SEGMENTS_CSV}...")
        df_recs = ml_optimizer.fit(PATH_USER_SEGMENTS_CSV)
        
        print("\nTiming Recommendations (Optimal Windows & Expected Engagement):")
        print(df_recs[['segment_name', 'optimal_window', 'user_fraction', 'expected_ctr']].head(10).to_string(index=False))
        
        # Export the final optimal windows to the output directory
        os.makedirs(os.path.dirname(PATH_OUTPUT_RECOMMENDATIONS), exist_ok=True)
        df_recs.to_csv(PATH_OUTPUT_RECOMMENDATIONS, index=False)
        df_recs.to_csv("../iteration_0_before_learning/timing_recommendations.csv", index=False)
        print(f"\nSuccessfully exported timing recommendations to: {PATH_OUTPUT_RECOMMENDATIONS}")
        
    except FileNotFoundError:
        print(f"Error: Could not find '{PATH_USER_SEGMENTS_CSV}'. Check the path.")
    except KeyError as e:
        print(f"Error: Column {e} not found in the CSV. Please verify column names.")

Loading data from ./output/users_segmented.csv...

Timing Recommendations (Optimal Windows & Expected Engagement):
    segment_name optimal_window  user_fraction  expected_ctr
Streak Champions        evening          0.600         0.363
Streak Champions          night          0.200         0.442
Streak Champions late_afternoon          0.200         0.521
  Power Learners    mid_morning          0.500         0.458
  Power Learners        evening          0.333         0.448
  Power Learners          night          0.167         0.570
 Top Competitors late_afternoon          1.000         0.421
 Coin Collectors        evening          0.400         0.364
 Coin Collectors  early_morning          0.200         0.304
 Steady Students        evening          0.545         0.388

Successfully exported timing recommendations to: ./output/timing_recommendations.csv


# Compute each user frequency


In [5]:
import pandas as pd
import numpy as np
import random

# ==========================================
# 1. Frequency Logic Definition
# ==========================================
def assign_daily_frequency(activity_score: float) -> int:
    """
    Takes a normalized activity score (0.0 to 1.0) and returns a deterministic
    daily notification limit based on defined thresholds.
    """
    # Handle edge cases (missing data)
    if pd.isna(activity_score):
        return 3
        
    if activity_score > 0.7:
        return 8
    elif 0.4 <= activity_score <= 0.7:
        return 5
    else:
        return 3

# ==========================================
# 2. Execution & Data Processing
# ==========================================
def run_frequency_optimizer(input_csv: str, output_csv: str):
    print(f"Loading user data from {input_csv}...")
    
    # Load your user data
    df_users = pd.read_csv(input_csv)
    
    score_column = None
    if 'activeness_score' in df_users.columns:
        score_column = 'activeness_score'
    elif 'activation_score' in df_users.columns:
        score_column = 'activation_score'
        print("Using 'activation_score' as the activity signal for notification limits.")
    else:
        raise KeyError("Missing both 'activeness_score' and 'activation_score'; cannot assign deterministic notification limits.")

    # Apply the frequency logic row-by-row
    df_users['daily_notification_limit'] = df_users[score_column].apply(assign_daily_frequency)
    
    # Show a quick summary of the distribution
    print("\nFrequency Distribution Summary:")
    print(df_users['daily_notification_limit'].value_counts().sort_index())
    
    # Save the updated dataframe
    df_users.to_csv(output_csv, index=False)
    print(f"\nSuccessfully exported updated user limits to: {output_csv}")

# ==========================================
# Run the pipeline
# ==========================================
if __name__ == "__main__":
    PATH_INPUT = './output/users_segmented.csv'
    PATH_OUTPUT = './output/users_frequency_assigned.csv'
    
    run_frequency_optimizer(PATH_INPUT, PATH_OUTPUT)

Loading user data from ./output/users_segmented.csv...
Using 'activation_score' as the activity signal for notification limits.

Frequency Distribution Summary:
daily_notification_limit
3    17
5    30
8    14
Name: count, dtype: int64

Successfully exported updated user limits to: ./output/users_frequency_assigned.csv
